# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [1]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-_tzikgwo
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-_tzikgwo
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Genera campioni di prova della wake word per verifica all'ascolto
target_word = 'ciao ernesto'

import os
from IPython.display import Audio

# Clona il generatore (se manca)
if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator

# Dipendenze: 'piper-tts' mancava ed e' quello che esegue la sintesi vera
!pip install -q torch torchaudio piper-phonemize-cross==1.2.1 onnxruntime piper-tts

# Scarica DUE voci Piper italiane (.onnx + .json) -- saltate se gia presenti
!mkdir -p piper-sample-generator/models
!wget -nc -O piper-sample-generator/models/it_IT-riccardo-x_low.onnx      'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx'
!wget -nc -O piper-sample-generator/models/it_IT-riccardo-x_low.onnx.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx.json'
!wget -nc -O piper-sample-generator/models/it_IT-paola-medium.onnx        'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx'
!wget -nc -O piper-sample-generator/models/it_IT-paola-medium.onnx.json   'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx.json'

# Genera 4 campioni alternando le due voci italiane
!cd piper-sample-generator && python3 -m piper_sample_generator "{target_word}" \
--model models/it_IT-riccardo-x_low.onnx \
--model models/it_IT-paola-medium.onnx \
--max-samples 4 \
--output-dir ../generated_samples

Audio(filename="generated_samples/0.wav", autoplay=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.5 MB/s eta 0:00:00
File ‘piper-sample-generator/models/it_IT-riccardo-x_low.onnx’ already there; not retrieving.
File ‘piper-sample-generator/models/it_IT-riccardo-x_low.onnx.json’ already there; not retrieving.
File ‘piper-sample-generator/models/it_IT-paola-medium.onnx’ already there; not retrieving.
File ‘piper-sample-generator/models/it_IT-paola-medium.onnx.json’ already there; not retrieving.
DEBUG:__main__:Loading ['models/it_IT-riccardo-x_low.onnx', 'models/it_IT-paola-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: models/it_IT-riccardo-x_low.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
DEBUG:piper.voice:Guessing voice config path: models/it_IT

In [6]:
from IPython.display import Audio, display

# Riccardo (maschile)
!cd piper-sample-generator && python3 -m piper_sample_generator "ciao ernesto" \
--model models/it_IT-riccardo-x_low.onnx --max-samples 1 --output-dir ../voce_riccardo

# Paola (femminile)
!cd piper-sample-generator && python3 -m piper_sample_generator "ciao ernesto" \
--model models/it_IT-paola-medium.onnx --max-samples 1 --output-dir ../voce_paola

print("RICCARDO (maschile):"); display(Audio(filename="voce_riccardo/0.wav"))
print("PAOLA (femminile):");   display(Audio(filename="voce_paola/0.wav"))

DEBUG:__main__:Loading ['models/it_IT-riccardo-x_low.onnx']
DEBUG:piper.voice:Guessing voice config path: models/it_IT-riccardo-x_low.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ciao ernesto, phonemes=[['t', 'ʃ', 'ˈ', 'a', 'o', ' ', 'e', 'ɾ', 'n', 'ˈ', 'ɛ', 's', 't', 'o']]
DEBUG:__main__:Loading ['models/it_IT-paola-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: models/it_IT-paola-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecut

PAOLA (femminile):


In [7]:
# Generazione in massa dei positivi (1000 campioni) con le due voci italiane
target_word = 'ciao ernesto'

!cd piper-sample-generator && python3 -m piper_sample_generator "{target_word}" \
--model models/it_IT-riccardo-x_low.onnx \
--model models/it_IT-paola-medium.onnx \
--max-samples 1000 \
--output-dir ../generated_samples

DEBUG:__main__:Loading ['models/it_IT-riccardo-x_low.onnx', 'models/it_IT-paola-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: models/it_IT-riccardo-x_low.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
DEBUG:piper.voice:Guessing voice config path: models/it_IT-paola-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ciao ernesto, phonemes=[['t', 'ʃ', 'ˈ', 'a', 'o', ' ', 'e', 'ɾ', 'n', 'ˈ', 'ɛ', 's', 't', 'o']]
DEBUG:piper.voice:text=ciao ernesto, phonemes=[['t', 'ʃ', 'ˈ', 'a', 'o', ' ', 'e', 'ɾ', 'n', 'ˈ', 'ɛ', 's', 't', 'o']]
DEBUG:piper.voice:text=ciao ernesto, phonemes=[['t', 'ʃ', 'ˈ', 'a', 'o', ' ', 'e', 'ɾ', 'n', 'ˈ', 'ɛ', 's', 't', 'o']]
DEBUG:pi

In [8]:
import glob
wavs = sorted(glob.glob("generated_samples/*.wav"))
print(f"Campioni positivi generati: {len(wavs)}")
print("Primi:", [w.split('/')[-1] for w in wavs[:3]])
print("Ultimi:", [w.split('/')[-1] for w in wavs[-3:]])

Campioni positivi generati: 1000
Primi: ['0.wav', '1.wav', '10.wav']
Ultimi: ['997.wav', '998.wav', '999.wav']


In [10]:
!pip install -q "datasets<4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.8 MB/s eta 0:00:00


In [1]:
import datasets; print(datasets.__version__)   # deve iniziare con 3.

3.6.0


In [2]:
!rm -rf mit_rirs audioset audioset_16k fma fma_16k

In [3]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [00:56,  4.82it/s]


--2026-06-04 19:59:03--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 18.239.50.103, 18.239.50.16, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.103|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-06-04 19:59:03 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]

--2026-06-04 19:59:03--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 18.239.50.103, 18.239.50.16, 18.239.50.49, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.103|:443... connected.
HTTP request sent, awaiting response... 

302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&X-Xet-Cas-Uid=public&Expires=1780606743&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4MDYwNjc0M319fV19&Signature=MEYCIQDJ7O%7EIfEv9YO55PoZxkp%7E6kaTWSDmbtZhZ6juBYl4mswIhAOKYNatjxTmxye5Z3vOR-r15J9yiWH9ebWXS4bUxwbu3&Key-Pair-Id=01KAYHXK2CBJSW0YZTMNXK9W1M [following]
--2026-06-04 19:59:03--  https://us.gcp.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b3364925998001

100%|██████████| 210/210 [00:21<00:00,  9.74it/s]


In [4]:
# === AudioSet (negativi) — nuovo formato Parquet, sostituisce il download .tar che dava 404 ===
import os, numpy as np, scipy.io.wavfile
from tqdm import tqdm
from datasets import load_dataset, Audio

output_dir = "audioset_16k"
os.makedirs(output_dir, exist_ok=True)

N = 2000  # numero di clip da estrarre (~quante ne aveva il vecchio shard .tar)

# streaming: NON scarica i 26 GB, legge solo quello che serve
ds = load_dataset("agkphysics/AudioSet", split="train", streaming=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))  # resample automatico 48k -> 16k

count = 0
for row in tqdm(ds, total=N):
    if count >= N:
        break
    arr = row["audio"]["array"]
    if arr is None or len(arr) == 0:
        continue
    arr16 = (np.clip(np.asarray(arr, dtype=np.float32), -1.0, 1.0) * 32767).astype(np.int16)
    scipy.io.wavfile.write(os.path.join(output_dir, f"audioset_{count:05d}.wav"), 16000, arr16)
    count += 1

print(f"Salvate {count} clip in {output_dir}/")

README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [01:33<00:00, 21.44it/s]

Salvate 2000 clip in audioset_16k/


In [5]:
# Setup augmentation. I percorsi sono già corretti per i dataset scaricati.
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.25,   # alzato da 0.1: piu' varieta' tonale (abbiamo solo 2 voci IT)
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )

In [6]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [7]:
# Augment samples and save the training, validation, and testing sets.
import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)

  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [8]:
# Scarica gli spettrogrammi pre-generati dei dataset negativi.
output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname
        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

--2026-06-04 20:17:26--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 18.239.50.49, 18.239.50.103, 18.239.50.16, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.49|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?Expires=1780607846&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY1ZTMyN2JjMTQ0NWE3NjhlZDM0M2I4Yy8yMjhkN2U3MmNkNWZkYzRlNmU1N2RhMzZiODhhNGMyMjdkMzRjYjhkYzQ0MDQxMDc4YjRjNGI2NWRjNzU4NDhkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MDYwNzg0Nn19fV19&Signature=MEYCIQDbBJiidUXzymIbrSj2Mze%7E8X5ucGz-EOw1drPazVeoPwIhAMOoWM31u8KTb4DBdp97FhXBCnf926S9r%7EulSHiZz-Hb&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-content-type=application%2Fzi

In [9]:
import os
print("=== Struttura negativi ===")
for d in ['speech','no_speech','dinner_party','dinner_party_eval']:
    p = f'negative_datasets/{d}'
    print(f"{p} -> {os.listdir(p) if os.path.exists(p) else 'MANCANTE'}")
print("\n=== Spazio disco ===")
!df -h /content | tail -1

=== Struttura negativi ===
negative_datasets/speech -> ['training']
negative_datasets/no_speech -> ['training']
negative_datasets/dinner_party -> ['training']
negative_datasets/dinner_party_eval -> ['validation_ambient', 'testing_ambient']

=== Spazio disco ===
overlay         113G   80G   34G  71% /


In [10]:
# Scrive il file di configurazione che controlla il training.
import yaml
import os

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/wakeword"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {  # Solo per validation e testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 128

config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]

config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500

config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

print("training_parameters.yaml scritto.")

training_parameters.yaml scritto.


In [12]:
# Patch train.py: in TF/Keras recenti model.evaluate(return_dict=True) restituisce gia'
# ndarray, quindi .numpy() sulle metriche crasha. Le avvolgiamo in np.asarray
# (no-op se sono gia' array, conversione se fossero tensori).
path = "/content/microWakeWord/microwakeword/train.py"
with open(path) as f:
    src = f.read()

repls = {
    'result["fp"].numpy()': 'np.asarray(result["fp"])',
    'ambient_predictions["tp"].numpy()': 'np.asarray(ambient_predictions["tp"])',
    'ambient_predictions["fp"].numpy()': 'np.asarray(ambient_predictions["fp"])',
    'ambient_predictions["fn"].numpy()': 'np.asarray(ambient_predictions["fn"])',
}
n = 0
for old, new in repls.items():
    c = src.count(old)
    src = src.replace(old, new)
    n += c
    print(f"{'OK' if c else 'NON TROVATO'}: {old}  (x{c})")

with open(path, "w") as f:
    f.write(src)

assert ".numpy()" not in src, "Restano chiamate .numpy() in train.py!"
print(f"\nPatch applicata: {n} sostituzioni, nessun .numpy() residuo.")

OK: result["fp"].numpy()  (x1)
OK: ambient_predictions["tp"].numpy()  (x1)
OK: ambient_predictions["fp"].numpy()  (x1)
OK: ambient_predictions["fn"].numpy()  (x1)

Patch applicata: 4 sostituzioni, nessun .numpy() residuo.


In [15]:
# Patch data.py: i set "ambient" sono enormi e venivano duplicati in RAM da uno
# shuffle inutile (data[indices] crea una copia) -> OOM su Colab. Per i set ambient
# restituiamo i dati senza copia, dimezzando il picco di memoria. L'ordine non conta
# per il conteggio dei falsi positivi.
path = "/content/microWakeWord/microwakeword/data.py"
with open(path) as f:
    src = f.read()

old = '''        indices = np.arange(labels.shape[0])

        if mode == "testing" or "validation":
            # Randomize the order of the data, weights, and labels
            np.random.shuffle(indices)

        return data[indices], labels[indices], weights[indices]'''

new = '''        if mode.endswith("_ambient"):
            # Set ambient enormi: niente shuffle/copia, dimezza il picco di RAM
            return data, labels, weights

        indices = np.arange(labels.shape[0])

        if mode in ("testing", "validation"):
            # Randomize the order of the data, weights, and labels
            np.random.shuffle(indices)

        return data[indices], labels[indices], weights[indices]'''

assert src.count(old) == 1, f"Blocco trovato {src.count(old)} volte (atteso 1)"
src = src.replace(old, new)
with open(path, "w") as f:
    f.write(src)
print("Patch data.py applicata: niente copia per i set ambient.")

Patch data.py applicata: niente copia per i set ambient.


In [17]:
import os, shutil
from mmap_ninja.ragged import RaggedMmap

base = "negative_datasets/dinner_party_eval"
FRACTION = 0.30  # tieni il 30%: ~1.1 GB invece di 3.79 GB

for mode in ["validation_ambient", "testing_ambient"]:
    mode_dir = os.path.join(base, mode)
    print(f"\n{mode_dir} contiene: {os.listdir(mode_dir)}")
    for entry in os.listdir(mode_dir):
        full = os.path.join(mode_dir, entry)
        if not os.path.isdir(full):
            continue
        try:
            rm = RaggedMmap(full)
            n = len(rm)
        except Exception as e:
            print(f"  '{entry}' non e' un RaggedMmap ({e}); salto")
            continue
        keep = max(1, int(n * FRACTION))
        tmp = full + "_small"
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
        RaggedMmap.from_generator(
            out_dir=tmp,
            sample_generator=(rm[i] for i in range(keep)),
            batch_size=100,
            verbose=False,
        )
        del rm
        shutil.rmtree(full)
        os.rename(tmp, full)
        print(f"  ridotto '{entry}': {n} -> {keep} spettrogrammi")

print("\nFatto. Set ambient ridotto a ~30%.")


negative_datasets/dinner_party_eval/validation_ambient contiene: ['chime6_dev_eval_mmap']
  ridotto 'chime6_dev_eval_mmap': 4 -> 1 spettrogrammi

negative_datasets/dinner_party_eval/testing_ambient contiene: ['dipco_u01_ch1_mmap']
  ridotto 'dipco_u01_ch1_mmap': 10 -> 3 spettrogrammi

Fatto. Set ambient ridotto a ~30%.


In [18]:
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
2026-06-04 20:58:43.740582: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780606723.742100   29086 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 204, 1, 

In [19]:
import os, glob
from google.colab import files

candidates = glob.glob("trained_models/wakeword/**/*.tflite", recursive=True)
print("File .tflite trovati:")
for c in candidates:
    print(f"  {c}  ({os.path.getsize(c)} byte)")

target = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
if not os.path.exists(target) and candidates:
    target = candidates[0]
print(f"\nScarico: {target}")
files.download(target)

File .tflite trovati:
  trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite  (60896 byte)

Scarico: trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>